### **This new notebook is for the 4 Features, 1 qubit per feature experiment, in total we have 4 Qubits. This notebook reutilizes a lot of code in QAOA_4F2Q_8Q.ipynb if you want to get a deeper insight you can refer to that notebook is well documented code**

**Imports**

In [ ]:
import math
from typing import Any

import kirin
from kirin.dialects import ilist
from bloqade import qasm2

**Variables for 4F1Q 4 Qubit in total experiment**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from itertools import product

# ========= Problem definition (4 assets / 4 qubits) =========
n_qubits = 4

asset_labels = [
    "A017 (Gov Bonds)",
    "A026 (IG Credit)",
    "A007 (Cash)",
    "A047 (HY Credit)",
]

# Ground truth from brute force
ground_truth_bits = "0011"
ground_truth_x = np.array([0, 0, 1, 1], dtype=int)
ground_truth_energy = -0.007414

# Ising local fields h
h_terms = np.array([
    -0.186803,   # A017
    -0.136982,   # A026
    -0.202042,   # A007
    -0.078664,   # A047
], dtype=float)

# Ising couplings J as symmetric matrix
J_terms = np.zeros((n_qubits, n_qubits), dtype=float)
J_terms[0,1] = J_terms[1,0] = 0.060001
J_terms[0,2] = J_terms[2,0] = 0.125001
J_terms[0,3] = J_terms[3,0] = 0.030001
J_terms[1,2] = J_terms[2,1] = 0.075000
J_terms[1,3] = J_terms[3,1] = 0.018001
J_terms[2,3] = J_terms[3,2] = 0.037501

# Constant shift (does not affect argmin, but useful for reporting)
ising_offset = 0.597754

print("n_qubits =", n_qubits)
print("ground_truth_bits =", ground_truth_bits)
print("ground_truth_energy =", ground_truth_energy)
print("h_terms =", h_terms)
print("J_terms =\n", J_terms)

**Calculate energy function**

In [ ]:
def bitstring_to_spin_array(bitstring: str) -> np.ndarray:
    # bit 0 -> s=-1, bit 1 -> s=+1
    return np.array([1 if b == "1" else -1 for b in bitstring], dtype=float)

def ising_energy_from_bitstring(bitstring: str, h_terms, J_terms, const=0.0) -> float:
    s = bitstring_to_spin_array(bitstring)
    e = 0.0

    # pair terms
    n = len(s)
    for i in range(n):
        for j in range(i + 1, n):
            e += J_terms[i, j] * s[i] * s[j]

    # local fields
    e += np.dot(h_terms, s)

    return float(e + const)

# quick sanity check
all_bitstrings = ["".join(bits) for bits in product("01", repeat=n_qubits)]
energy_table = pd.DataFrame({
    "bitstring": all_bitstrings,
    "energy": [ising_energy_from_bitstring(bs, h_terms, J_terms, ising_offset) for bs in all_bitstrings]
}).sort_values("energy", ascending=True).reset_index(drop=True)

energy_table.head(10)

**Helper functions to analyze the data**

In [ ]:
def normalize_prob_dict(prob_dict):
    total = sum(prob_dict.values())
    if total == 0:
        return prob_dict
    return {k: v / total for k, v in prob_dict.items()}

def expected_energy_from_probs(prob_dict, h_terms, J_terms, const=0.0):
    prob_dict = normalize_prob_dict(prob_dict)
    e = 0.0
    for bitstring, p in prob_dict.items():
        e += p * ising_energy_from_bitstring(bitstring, h_terms, J_terms, const)
    return float(e)

def ground_truth_probability(prob_dict, ground_truth_bits):
    prob_dict = normalize_prob_dict(prob_dict)
    return float(prob_dict.get(ground_truth_bits, 0.0))

def marginal_one_probs(prob_dict, n_qubits):
    prob_dict = normalize_prob_dict(prob_dict)
    marginals = np.zeros(n_qubits, dtype=float)

    for bitstring, p in prob_dict.items():
        for i, b in enumerate(bitstring):
            if b == "1":
                marginals[i] += p

    return marginals

def top_states_from_probs(prob_dict, top_k=10):
    prob_dict = normalize_prob_dict(prob_dict)
    rows = []
    for bitstring, p in prob_dict.items():
        rows.append({
            "bitstring": bitstring,
            "probability": p,
            "energy": ising_energy_from_bitstring(bitstring, h_terms, J_terms, ising_offset)
        })
    df = pd.DataFrame(rows).sort_values("probability", ascending=False).reset_index(drop=True)
    return df.head(top_k)

**convert ising data to format expected**

In [ ]:
# Convert current Ising data to the format expected by build_qaoa_ising_kernel

h_terms_list = [(i, float(h_terms[i])) for i in range(n_qubits)]

J_terms_list = []
for i in range(n_qubits):
    for j in range(i + 1, n_qubits):
        if abs(J_terms[i, j]) > 1e-12:
            J_terms_list.append((i, j, float(J_terms[i, j])))

print("h_terms_list =", h_terms_list)
print("J_terms_list =", J_terms_list)

### **Construct the circuit**

In [ ]:
def build_qaoa_ising_kernel(n_qubits, h_terms, J_terms):
    @qasm2.extended
    def kernel(gamma: ilist.IList[float, Any], beta: ilist.IList[float, Any]):
        q = qasm2.qreg(n_qubits)

        for i in range(n_qubits):
            qasm2.h(q[i])

        for layer in range(len(gamma)):
            g = gamma[layer]
            b = beta[layer]

            for k in range(len(h_terms)):
                term = h_terms[k]
                i = term[0]
                hi = term[1]
                qasm2.rz(q[i], 2.0 * g * hi)

            for k in range(len(J_terms)):
                term = J_terms[k]
                i = term[0]
                j = term[1]
                Jij = term[2]

                qasm2.cx(q[i], q[j])
                qasm2.rz(q[j], 2.0 * g * Jij)
                qasm2.cx(q[i], q[j])

            for i in range(n_qubits):
                qasm2.rx(q[i], 2.0 * b)

        return q

    return kernel

In [ ]:
from scipy.optimize import minimize
from bloqade.pyqrack import StackMemorySimulator
import numpy as np

# =========================================================
# CLEAN QAOA INFERENCE CELL FOR 4 ASSETS / 4 QUBITS
# Uses the numeric h_terms and J_terms already defined above
# =========================================================

# ---------- make naming consistent ----------
ground_truth_bitstring = ground_truth_bits

# ---------- tunable config ----------
P = 1
MAXITER = 5000
RHOBEG = 0.2
METHOD = "COBYLA"

INITIAL_POINTS = [
    ([0.3] * P, [0.2] * P),
    ([0.5] * P, [0.5] * P),
    ([1.0] * P, [0.5] * P),
    ([2.0] * P, [1.0] * P),
    ([np.pi / 2] * P, [np.pi / 4] * P),
]

TOP_K = 15
SHOTS = 2000
SEED = 123
VERBOSE = False

# ---------- numeric Hamiltonian ----------
h_numeric = np.asarray(h_terms).reshape(-1).astype(float)
J_numeric = np.asarray(J_terms).astype(float)

print("h_numeric shape:", h_numeric.shape)
print("J_numeric shape:", J_numeric.shape)

# ---------- Bloqade-format terms ----------
h_terms_list = [(i, float(h_numeric[i])) for i in range(n_qubits)]

J_terms_list = []
for i in range(n_qubits):
    for j in range(i + 1, n_qubits):
        if abs(J_numeric[i, j]) > 1e-12:
            J_terms_list.append((i, j, float(J_numeric[i, j])))

# ---------- build kernel + simulator ----------
kernel = build_qaoa_ising_kernel(
    n_qubits=n_qubits,
    h_terms=h_terms_list,
    J_terms=J_terms_list
)

sim = StackMemorySimulator(min_qubits=n_qubits)

# ---------- helper functions ----------
def join_theta(gamma_vals, beta_vals):
    return np.array(list(gamma_vals) + list(beta_vals), dtype=float)

def split_theta(theta, p):
    theta = np.array(theta, dtype=float)
    gamma_vals = theta[:p].tolist()
    beta_vals = theta[p:2*p].tolist()
    return gamma_vals, beta_vals

def ising_energy_from_x_list_local(x_list, h_numeric, J_numeric, const=0.0):
    x = np.array(x_list, dtype=int)
    s = 2 * x - 1
    e = 0.0

    for i in range(len(s)):
        e += float(h_numeric[i]) * s[i]

    for i in range(len(s)):
        for j in range(i + 1, len(s)):
            e += float(J_numeric[i, j]) * s[i] * s[j]

    return float(e + const)

def probs_from_theta(theta):
    gamma_vals, beta_vals = split_theta(theta, P)
    ket = sim.state_vector(kernel, args=(gamma_vals, beta_vals))
    probs = np.abs(np.array(ket)) ** 2
    probs = probs / probs.sum()
    return probs

def expected_ising_energy_for_theta(theta):
    probs = probs_from_theta(theta)

    exp_energy = 0.0
    for idx in range(2 ** n_qubits):
        bitstring = format(idx, f"0{n_qubits}b")
        x_list = [int(b) for b in bitstring]
        energy = ising_energy_from_x_list_local(x_list, h_numeric, J_numeric, ising_offset)
        exp_energy += probs[idx] * energy

    return float(exp_energy)

def optimize_once(theta0):
    eval_history = []

    def objective(theta):
        value = expected_ising_energy_for_theta(theta)
        eval_history.append((theta.copy(), value))
        if VERBOSE:
            g, b = split_theta(theta, P)
            print(f"gamma={np.round(g, 6)}, beta={np.round(b, 6)}, expected_energy={value:.8f}")
        return value

    result = minimize(
        objective,
        theta0,
        method=METHOD,
        options={"maxiter": MAXITER, "rhobeg": RHOBEG}
    )
    return result, eval_history

def top_states_from_probs_array(probs, top_k=15):
    top_indices = np.argsort(probs)[::-1][:top_k]
    rows = []

    for rank, idx in enumerate(top_indices, start=1):
        bitstring = format(idx, f"0{n_qubits}b")
        x_list = [int(b) for b in bitstring]
        energy = ising_energy_from_x_list_local(x_list, h_numeric, J_numeric, ising_offset)
        rows.append({
            "rank": rank,
            "bitstring": bitstring,
            "prob": float(probs[idx]),
            "energy": float(energy),
        })
    return rows

def sample_counts_from_probs(probs, shots=1000, seed=123):
    rng = np.random.default_rng(seed)
    indices = np.arange(len(probs))
    probs = np.array(probs, dtype=float)
    probs = probs / probs.sum()
    sampled = rng.choice(indices, size=shots, p=probs)

    counts = {}
    for idx in sampled:
        bitstring = format(idx, f"0{n_qubits}b")
        counts[bitstring] = counts.get(bitstring, 0) + 1

    return dict(sorted(counts.items(), key=lambda kv: kv[1], reverse=True))

# ---------- multi-start classical optimization ----------
all_runs = []
best_result = None
best_history = None
best_theta0 = None

for run_id, (gamma0, beta0) in enumerate(INITIAL_POINTS, start=1):
    theta0 = join_theta(gamma0, beta0)
    result, history = optimize_once(theta0)

    all_runs.append({
        "run_id": run_id,
        "theta0": theta0,
        "result": result,
        "history": history
    })

    if best_result is None or result.fun < best_result.fun:
        best_result = result
        best_history = history
        best_theta0 = theta0

# ---------- recover best params ----------
best_gamma, best_beta = split_theta(best_result.x, P)

print("===================================")
print("BEST RUN SUMMARY")
print("===================================")
print("Initial theta:", best_theta0)
print("Best theta:", best_result.x)
print("Best gamma:", best_gamma)
print("Best beta:", best_beta)
print("Best expected energy:", best_result.fun)
print("Success:", best_result.success)
print("Message:", best_result.message)

# ---------- evaluate final optimized circuit ----------
ket_opt = sim.state_vector(kernel, args=(best_gamma, best_beta))
probs_opt = np.abs(np.array(ket_opt)) ** 2
probs_opt = probs_opt / probs_opt.sum()

# ---------- top states ----------
rows = top_states_from_probs_array(probs_opt, top_k=TOP_K)

print("\n===================================")
print(f"TOP {TOP_K} STATES AFTER OPTIMIZATION")
print("===================================")
for r in rows:
    print(f"{r['rank']:2d}. bitstring={r['bitstring']}   prob={r['prob']:.8f}   energy={r['energy']:.8f}")

# ---------- ground truth info ----------
gt_index = int(ground_truth_bitstring, 2)
gt_energy = ising_energy_from_x_list_local(
    [int(b) for b in ground_truth_bitstring],
    h_numeric,
    J_numeric,
    ising_offset
)

print("\n===================================")
print("GROUND TRUTH CHECK")
print("===================================")
print("Ground-truth bitstring:", ground_truth_bitstring)
print("Ground-truth probability:", probs_opt[gt_index])
print("Ground-truth energy:", gt_energy)

# ---------- shot-based sampling ----------
counts = sample_counts_from_probs(probs_opt, shots=SHOTS, seed=SEED)

print("\n===================================")
print(f"TOP 10 SAMPLED STATES ({SHOTS} shots)")
print("===================================")
for rank, (bitstring, count) in enumerate(list(counts.items())[:10], start=1):
    energy = ising_energy_from_x_list_local(
        [int(b) for b in bitstring],
        h_numeric,
        J_numeric,
        ising_offset
    )
    print(f"{rank:2d}. bitstring={bitstring}   count={count}   freq={count/SHOTS:.6f}   energy={energy:.8f}")

In [ ]:
import numpy as np

# --- normalize h_terms into numeric 1D vector ---
if isinstance(h_terms, np.ndarray):
    h_arr = np.array(h_terms, dtype=object)
else:
    h_arr = np.array(h_terms, dtype=object)

# Case 1: already numeric 1D
if h_arr.ndim == 1 and all(np.isscalar(x) for x in h_arr):
    h_numeric = h_arr.astype(float)

# Case 2: list/array of pairs like [(0,h0), (1,h1), ...]
elif h_arr.ndim == 2 and h_arr.shape[1] == 2:
    h_numeric = h_arr[:, 1].astype(float)

# Fallback for tuples in 1D object array
elif h_arr.ndim == 1 and all(isinstance(x, (tuple, list, np.ndarray)) and len(x) == 2 for x in h_arr):
    h_numeric = np.array([x[1] for x in h_arr], dtype=float)

else:
    raise ValueError(f"Unsupported h_terms format. shape={h_arr.shape}, value={h_terms}")

# --- normalize J_terms into numeric NxN matrix ---
J_arr = np.array(J_terms, dtype=object)

# Case 1: already numeric matrix
if J_arr.ndim == 2 and J_arr.shape[0] == J_arr.shape[1]:
    J_numeric = J_arr.astype(float)

# Case 2: list of triples like [(i,j,Jij), ...]
elif J_arr.ndim == 2 and J_arr.shape[1] == 3:
    J_numeric = np.zeros((n_qubits, n_qubits), dtype=float)
    for i, j, Jij in J_arr:
        i, j = int(i), int(j)
        J_numeric[i, j] = float(Jij)
        J_numeric[j, i] = float(Jij)

# Fallback for 1D object array of triples
elif J_arr.ndim == 1 and all(isinstance(x, (tuple, list, np.ndarray)) and len(x) == 3 for x in J_arr):
    J_numeric = np.zeros((n_qubits, n_qubits), dtype=float)
    for i, j, Jij in J_arr:
        i, j = int(i), int(j)
        J_numeric[i, j] = float(Jij)
        J_numeric[j, i] = float(Jij)

else:
    raise ValueError(f"Unsupported J_terms format. shape={J_arr.shape}, value={J_terms}")

# --- Bloqade-format terms for kernel ---
h_terms_list = [(i, float(h_numeric[i])) for i in range(n_qubits)]

J_terms_list = []
for i in range(n_qubits):
    for j in range(i + 1, n_qubits):
        if abs(J_numeric[i, j]) > 1e-12:
            J_terms_list.append((i, j, float(J_numeric[i, j])))

kernel = build_qaoa_ising_kernel(
    n_qubits=n_qubits,
    h_terms=h_terms_list,
    J_terms=J_terms_list
)

print("h_numeric =", h_numeric)
print("J_numeric =\n", J_numeric)
print("h_terms_list =", h_terms_list)
print("J_terms_list =", J_terms_list)
print("QAOA kernel construido.")

### **Inference cell**

In [ ]:
from scipy.optimize import minimize
from bloqade.pyqrack import StackMemorySimulator
import numpy as np

# =========================================================
# QAOA INFERENCE CELL FOR 4 ASSETS / 4 QUBITS
# Assumes these were already defined in previous cells:
# - n_qubits
# - ground_truth_bitstring
# - ising_offset
# - h_terms        -> numeric numpy array, shape (n_qubits,)
# - J_terms        -> numeric numpy array, shape (n_qubits, n_qubits)
# - build_qaoa_ising_kernel(...)
# =========================================================

# ---------- tunable config ----------
P = 1
MAXITER = 5000
RHOBEG = 0.2
METHOD = "COBYLA"

INITIAL_POINTS = [
    ([0.3] * P, [0.2] * P),
    ([0.5] * P, [0.5] * P),
    ([1.0] * P, [0.5] * P),
    ([2.0] * P, [1.0] * P),
    ([np.pi / 2] * P, [np.pi / 4] * P),
]

TOP_K = 15
SHOTS = 2000
SEED = 123
VERBOSE = False

# ---------- make sure numeric Hamiltonian objects are numeric ----------
h_numeric = np.array(h_terms, dtype=float)
J_numeric = np.array(J_terms, dtype=float)

# ---------- Bloqade-format terms for the kernel ----------
h_terms_list = [(i, float(h_numeric[i])) for i in range(n_qubits)]

J_terms_list = []
for i in range(n_qubits):
    for j in range(i + 1, n_qubits):
        if abs(J_numeric[i, j]) > 1e-12:
            J_terms_list.append((i, j, float(J_numeric[i, j])))

# ---------- build kernel + simulator ----------
kernel = build_qaoa_ising_kernel(
    n_qubits=n_qubits,
    h_terms=h_terms_list,
    J_terms=J_terms_list
)

sim = StackMemorySimulator(min_qubits=n_qubits)

# ---------- helper functions ----------
def join_theta(gamma_vals, beta_vals):
    return np.array(list(gamma_vals) + list(beta_vals), dtype=float)

def split_theta(theta, p):
    theta = np.array(theta, dtype=float)
    gamma_vals = theta[:p].tolist()
    beta_vals = theta[p:2*p].tolist()
    return gamma_vals, beta_vals

def ising_energy_from_x_list(x_list, h_numeric, J_numeric, const=0.0):
    x = np.array(x_list, dtype=int)
    s = 2 * x - 1  # x in {0,1} -> s in {-1,+1}
    e = 0.0

    for i in range(len(s)):
        e += float(h_numeric[i]) * s[i]

    for i in range(len(s)):
        for j in range(i + 1, len(s)):
            e += float(J_numeric[i, j]) * s[i] * s[j]

    return float(e + const)

def probs_from_theta(theta):
    gamma_vals, beta_vals = split_theta(theta, P)

    ket = sim.state_vector(
        kernel,
        args=(gamma_vals, beta_vals)
    )
    probs = np.abs(np.array(ket)) ** 2
    probs = probs / probs.sum()
    return probs

def expected_ising_energy_for_theta(theta):
    probs = probs_from_theta(theta)

    exp_energy = 0.0
    for idx in range(2 ** n_qubits):
        bitstring = format(idx, f"0{n_qubits}b")
        x_list = [int(b) for b in bitstring]
        energy = ising_energy_from_x_list(x_list, h_numeric, J_numeric, ising_offset)
        exp_energy += probs[idx] * energy

    return float(exp_energy)

def optimize_once(theta0):
    eval_history = []

    def objective(theta):
        value = expected_ising_energy_for_theta(theta)
        eval_history.append((theta.copy(), value))
        if VERBOSE:
            g, b = split_theta(theta, P)
            print(f"gamma={np.round(g, 6)}, beta={np.round(b, 6)}, expected_energy={value:.8f}")
        return value

    result = minimize(
        objective,
        theta0,
        method=METHOD,
        options={"maxiter": MAXITER, "rhobeg": RHOBEG}
    )
    return result, eval_history

def top_states_from_probs_array(probs, top_k=15):
    top_indices = np.argsort(probs)[::-1][:top_k]
    rows = []

    for rank, idx in enumerate(top_indices, start=1):
        bitstring = format(idx, f"0{n_qubits}b")
        x_list = [int(b) for b in bitstring]
        energy = ising_energy_from_x_list(x_list, h_numeric, J_numeric, ising_offset)
        rows.append({
            "rank": rank,
            "bitstring": bitstring,
            "prob": float(probs[idx]),
            "energy": float(energy),
        })
    return rows

def sample_counts_from_probs(probs, shots=1000, seed=123):
    rng = np.random.default_rng(seed)
    indices = np.arange(len(probs))
    probs = np.array(probs, dtype=float)
    probs = probs / probs.sum()
    sampled = rng.choice(indices, size=shots, p=probs)

    counts = {}
    for idx in sampled:
        bitstring = format(idx, f"0{n_qubits}b")
        counts[bitstring] = counts.get(bitstring, 0) + 1

    return dict(sorted(counts.items(), key=lambda kv: kv[1], reverse=True))

# ---------- multi-start classical optimization ----------
all_runs = []
best_result = None
best_history = None
best_theta0 = None

for run_id, (gamma0, beta0) in enumerate(INITIAL_POINTS, start=1):
    theta0 = join_theta(gamma0, beta0)
    result, history = optimize_once(theta0)

    all_runs.append({
        "run_id": run_id,
        "theta0": theta0,
        "result": result,
        "history": history
    })

    if best_result is None or result.fun < best_result.fun:
        best_result = result
        best_history = history
        best_theta0 = theta0

# ---------- recover best params ----------
best_gamma, best_beta = split_theta(best_result.x, P)

print("===================================")
print("BEST RUN SUMMARY")
print("===================================")
print("Initial theta:", best_theta0)
print("Best theta:", best_result.x)
print("Best gamma:", best_gamma)
print("Best beta:", best_beta)
print("Best expected energy:", best_result.fun)
print("Success:", best_result.success)
print("Message:", best_result.message)

# ---------- evaluate final optimized circuit ----------
ket_opt = sim.state_vector(
    kernel,
    args=(best_gamma, best_beta)
)
probs_opt = np.abs(np.array(ket_opt)) ** 2
probs_opt = probs_opt / probs_opt.sum()

# ---------- top states ----------
rows = top_states_from_probs_array(probs_opt, top_k=TOP_K)

print("\n===================================")
print(f"TOP {TOP_K} STATES AFTER OPTIMIZATION")
print("===================================")
for r in rows:
    print(f"{r['rank']:2d}. bitstring={r['bitstring']}   prob={r['prob']:.8f}   energy={r['energy']:.8f}")

# ---------- ground truth info ----------
gt_index = int(ground_truth_bitstring, 2)
gt_energy = ising_energy_from_x_list(
    [int(b) for b in ground_truth_bitstring],
    h_numeric,
    J_numeric,
    ising_offset
)

print("\n===================================")
print("GROUND TRUTH CHECK")
print("===================================")
print("Ground-truth bitstring:", ground_truth_bitstring)
print("Ground-truth probability:", probs_opt[gt_index])
print("Ground-truth energy:", gt_energy)

# ---------- shot-based sampling ----------
counts = sample_counts_from_probs(probs_opt, shots=SHOTS, seed=SEED)

print("\n===================================")
print(f"TOP 10 SAMPLED STATES ({SHOTS} shots)")
print("===================================")
for rank, (bitstring, count) in enumerate(list(counts.items())[:10], start=1):
    energy = ising_energy_from_x_list(
        [int(b) for b in bitstring],
        h_numeric,
        J_numeric,
        ising_offset
    )
    print(f"{rank:2d}. bitstring={bitstring}   count={count}   freq={count/SHOTS:.6f}   energy={energy:.8f}")

**Summary Table Function**

In [ ]:
summary_df = pd.DataFrame([
    {
        "P": r["P"],
        "runtime_sec": r["runtime_sec"],
        "best_fun": r["best_fun"],
        "expected_energy": r["expected_energy"],
        "ground_truth_probability": r["ground_truth_probability"],
    }
    for r in experiment_results
])

summary_df.sort_values("P").reset_index(drop=True)

**1ST Graph, Expected energy vs P**

In [ ]:
plot_df = summary_df.sort_values("P")

plt.figure(figsize=(7,5))
plt.plot(plot_df["P"], plot_df["expected_energy"], marker="o")
plt.axhline(ground_truth_energy, linestyle="--")
plt.xlabel("QAOA depth P")
plt.ylabel("Expected energy")
plt.title("Best expected energy vs P (4 assets / 4 qubits)")
plt.grid(True, alpha=0.3)
plt.show()

**2ST Graph, Runtime vs P**

In [ ]:
plt.figure(figsize=(7,5))
plt.plot(plot_df["P"], plot_df["runtime_sec"], marker="o")
plt.xlabel("QAOA depth P")
plt.ylabel("Runtime (sec)")
plt.title("Runtime vs P (4 assets / 4 qubits)")
plt.grid(True, alpha=0.3)
plt.show()

**3RD Graph, Ground Truth probability VS P**

In [ ]:
plt.figure(figsize=(7,5))
plt.plot(plot_df["P"], plot_df["ground_truth_probability"], marker="o")
plt.xlabel("QAOA depth P")
plt.ylabel(f'Probability of ground truth state {ground_truth_bits}')
plt.title("Ground-truth probability vs P (4 assets / 4 qubits)")
plt.grid(True, alpha=0.3)
plt.show()

**Best run and top states table**

In [ ]:
best_idx = summary_df["ground_truth_probability"].idxmax()
best_run = experiment_results[best_idx]

print("Best run by ground-truth probability:")
print("P =", best_run["P"])
print("best_gamma =", best_run["best_gamma"])
print("best_beta =", best_run["best_beta"])
print("expected_energy =", best_run["expected_energy"])
print("ground_truth_probability =", best_run["ground_truth_probability"])

top_states_from_probs(best_run["probs_opt"], top_k=10)

**1ST Qubit individual probability graph**

In [ ]:
marginals = marginal_one_probs(best_run["probs_opt"], n_qubits)

plt.figure(figsize=(8,5))
plt.bar(range(n_qubits), marginals)
plt.xticks(range(n_qubits), asset_labels, rotation=25, ha="right")
plt.ylabel("P(qubit = 1)")
plt.title(f"Marginal probability of measuring 1 (best run, P={best_run['P']})")
plt.tight_layout()
plt.show()

**Graph 5:**

In [ ]:
top_df = top_states_from_probs(best_run["probs_opt"], top_k=16).sort_values("probability", ascending=False)

plt.figure(figsize=(10,5))
plt.bar(top_df["bitstring"], top_df["probability"])
plt.xlabel("Bitstring")
plt.ylabel("Probability")
plt.title(f"Measured state distribution (best run, P={best_run['P']})")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()